In [0]:
# MAGIC %md
# MAGIC ### Step 1: Initialize Workflow Parameters & Catalog
# MAGIC Standardizing parameters for workflow orchestration.

dbutils.widgets.text("catalog_name", "smart-factory-catalog", "Unity Catalog Name")
dbutils.widgets.text("silver_schema", "silver", "Silver Schema Name")
dbutils.widgets.text("gold_schema", "gold", "Gold Schema Name")
dbutils.widgets.text("base_s3_path", "s3://smart-factory-analytics-bucket", "Base S3 Path")

catalog_name = dbutils.widgets.get("catalog_name")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")
base_s3_path = dbutils.widgets.get("base_s3_path")

checkpoint_path = f"{base_s3_path}/unity-catalog/checkpoints/{gold_schema}/hourly_machine_summary/"

# Ensure Gold schema exists
spark.sql(f"USE CATALOG `{catalog_name}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{gold_schema}`")

In [0]:
# MAGIC %md
# MAGIC ### Step 2: Define Target Gold Table
# MAGIC Creating the aggregated reporting table.

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{catalog_name}`.`{gold_schema}`.hourly_machine_summary (
    device_id STRING,
    device_type STRING,
    zone STRING,
    window_start TIMESTAMP,
    window_end TIMESTAMP,
    avg_temperature_c DOUBLE,
    peak_vibration_x DOUBLE,
    total_power_kwh DOUBLE,
    critical_alert_count BIGINT,
    last_updated TIMESTAMP
)
USING DELTA
LOCATION '{base_s3_path}/unity-catalog/tables/{gold_schema}/hourly_machine_summary/'
""")

In [0]:
# MAGIC %md
# MAGIC ### Step 3: Read Change Data Feed (CDF) from Silver
# MAGIC We read strictly the net-new inserts from all three Silver tables and union them into a single logical stream.

from pyspark.sql.functions import col, window, max, avg, sum, when, current_timestamp
from delta.tables import DeltaTable

def read_silver_cdf(stream_name: str):
    return (spark.readStream
        .format("delta")
        .option("readChangeFeed", "true")
        # Ignore updates/deletes, we only want to aggregate net-new sensor facts
        .option("ignoreDeletes", "true") 
        .option("ignoreChanges", "true")
        .table(f"`{catalog_name}`.`{silver_schema}`.{stream_name}_telemetry_clean")
        .filter(col("_change_type") == "insert")
    )

print("Initializing Silver CDF Streams...")
df_cnc = read_silver_cdf("cnc")
df_conveyor = read_silver_cdf("conveyor")
df_temp = read_silver_cdf("temperature")

# Union all three streams into a unified factory stream
df_factory_unified = df_cnc.union(df_conveyor).union(df_temp)

In [0]:
# MAGIC %md
# MAGIC ### Step 4: Time-Window Aggregations
# MAGIC Calculate business-critical metrics aggregated by machine and hour.

df_aggregated = (df_factory_unified
    # Watermarking handles late-arriving data (e.g., a machine lost WiFi for 30 mins)
    .withWatermark("event_time", "1 hour")
    .groupBy(
        "device_id", 
        "device_type", 
        "zone",
        window("event_time", "1 hour")
    )
    .agg(
        avg("temperature_c").alias("avg_temperature_c"),
        max("vibration_x").alias("peak_vibration_x"),
        # Convert total Estimated Watts into Kilowatt Hours (kWh) for the hour
        (sum("est_power_watts") / 1000).alias("total_power_kwh"),
        # Count how many times this machine hit 'CRITICAL' status in this hour
        sum(when(col("machine_health_status") == "CRITICAL", 1).otherwise(0)).alias("critical_alert_count")
    )
    .withColumn("last_updated", current_timestamp())
)

In [0]:
# MAGIC %md
# MAGIC ### Step 5: Stateful Upsert into Gold
# MAGIC As the 1-hour window receives new late data, we use MERGE to safely overwrite the row in the Gold table without duplicating the hour.

def upsert_to_gold(microBatchDF, batchId):
    target_table = f"`{catalog_name}`.`{gold_schema}`.hourly_machine_summary"
    target_delta_table = DeltaTable.forName(spark, target_table)
    
    # Flatten the struct created by the window() function
    df_clean_batch = microBatchDF.selectExpr(
        "device_id", "device_type", "zone",
        "window.start as window_start", 
        "window.end as window_end",
        "avg_temperature_c", "peak_vibration_x", "total_power_kwh", 
        "critical_alert_count", "last_updated"
    )
    
    (target_delta_table.alias("target")
        .merge(
            df_clean_batch.alias("source"),
            # The Composite Primary Key: We update if the Machine AND the Hour match
            "target.device_id = source.device_id AND target.window_start = source.window_start"
        )
        # For Gold, we DO use matched updates so rolling calculations can update live!
        .whenMatchedUpdateAll() 
        .whenNotMatchedInsertAll()
        .execute()
    )

print("Starting Gold Aggregation Pipeline...")

try:
    gold_stream = (df_aggregated.writeStream
        .foreachBatch(upsert_to_gold)
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True) # Runs as a micro-batch
        .start()
    )
    
    gold_stream.awaitTermination()
    
    print("Optimizing Gold Dashboard Table...")
    spark.sql(f"OPTIMIZE `{catalog_name}`.`{gold_schema}`.hourly_machine_summary ZORDER BY (zone, device_type)")
    
    print("✅ Gold pipeline completed successfully.")

except Exception as e:
    print(f"❌ CRITICAL PIPELINE FAILURE: {str(e)}")
    raise e